# 15. 错误与异常

错误与异常处理用于让程序在遇到问题时不至于直接崩溃，并且可以给出更清晰的处理逻辑。

本章重点内容：

- 常见错误类型
- `try...except`
- 捕获多个异常
- `else` 和 `finally`
- 主动抛出异常：`raise`
- 自定义异常
- 异常信息和 traceback
- 异常处理的常见误区

本章不包含练习题，只保留学习笔记和示例代码。


## 1. 错误类型概览

程序中的问题大致可以分成三类：

- 语法错误：代码不符合 Python 语法，程序无法运行。
- 运行时异常：语法没问题，但运行过程中出错。
- 逻辑错误：程序能运行，但结果不符合预期。


In [1]:
# 语法错误示例：
# if True
#     print('缺少冒号')

# 运行时异常示例：
# print(10 / 0)              # ZeroDivisionError
# print(int('abc'))          # ValueError
# print([1, 2, 3][10])       # IndexError
# print({'name': 'Tom'}['age'])  # KeyError

# 逻辑错误示例：
price = 100
discount = 20

# 如果 discount 表示 8 折，就不应该写成 price - discount。
# 这段代码不会报错，但业务含义可能是错的。
final_price = price - discount
print(final_price)


80


### 解释

- 语法错误必须先改代码，不能靠异常处理解决。
- 运行时异常可以用 `try...except` 捕获和处理。
- 逻辑错误最难发现，需要通过测试、检查需求和打印中间结果来定位。


## 2. `try...except` 基本写法

当你知道某段代码可能出错，但希望程序继续运行时，可以使用 `try...except`。


In [2]:
text = 'abc'

try:
    # 尝试把字符串转成整数
    number = int(text)
    print(number)
except ValueError:
    # 如果 int(text) 失败，就执行这里
    print('转换失败：文本不是有效整数')

print('程序继续执行')


转换失败：文本不是有效整数
程序继续执行


### 解释

- `try` 代码块放可能出错的代码。
- `except` 代码块放出错后的处理逻辑。
- 捕获异常后，程序不会因为这个异常直接中断。
- 不建议用异常处理隐藏本来应该修复的代码错误。


## 3. 捕获多个异常

一段代码可能出现不同类型的异常，可以分别捕获并给出不同处理。


In [3]:
data = {'count': '0'}

try:
    count = int(data['count'])
    result = 100 / count
    print(result)
except KeyError as e:
    print('缺少字段：', e)
except ValueError as e:
    print('数字格式错误：', e)
except ZeroDivisionError as e:
    print('除数不能为 0：', e)


try:
    value = int('abc')
except (ValueError, TypeError) as e:
    # 多个异常处理方式相同时，可以写在一个元组里
    print('转换失败：', e)


除数不能为 0： division by zero
转换失败： invalid literal for int() with base 10: 'abc'


### 解释

- `except KeyError as e` 可以把异常对象保存到变量 `e` 中。
- 多个 `except` 会从上往下匹配，匹配到第一个合适的分支后停止。
- 更具体的异常应该写在更前面，范围大的异常写在后面。


## 4. `else` 和 `finally`

异常处理中也可以使用 `else` 和 `finally`。

- `else`：没有异常时执行。
- `finally`：无论是否异常都会执行。


In [4]:
filename = 'demo.txt'

try:
    print('准备执行可能出错的代码')
    number = int('123')
except ValueError:
    print('转换失败')
else:
    # try 中没有异常才执行
    print('转换成功：', number)
finally:
    # 无论是否出错都会执行
    print('清理动作：关闭连接、释放资源等')


准备执行可能出错的代码
转换成功： 123
清理动作：关闭连接、释放资源等


### 解释

- `else` 可以放成功路径的逻辑，让 `try` 代码块更短。
- `finally` 常用于资源清理，例如关闭文件、关闭网络连接、释放锁。
- 即使 `try` 或 `except` 中有 `return`，`finally` 通常仍会执行。


## 5. 主动抛出异常：`raise`

当函数发现参数或状态不符合要求时，可以主动抛出异常，让调用者知道问题在哪里。


In [5]:
def set_age(age):
    if age < 0:
        # 主动抛出异常，说明这个参数非法
        raise ValueError('年龄不能为负数')

    print('年龄设置成功：', age)


set_age(18)

try:
    set_age(-3)
except ValueError as e:
    print('捕获到异常：', e)


年龄设置成功： 18
捕获到异常： 年龄不能为负数


### 解释

- `raise ValueError(...)` 表示主动抛出值错误。
- 抛出异常后，当前函数会中断。
- 调用者可以选择捕获异常，也可以让异常继续向上抛出。


## 6. 自定义异常

当内置异常不能准确表达业务问题时，可以定义自己的异常类。


In [6]:
class InsufficientBalanceError(Exception):
    pass


class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientBalanceError('余额不足，无法取款')

        self.balance -= amount
        return self.balance


account = Account(100)

try:
    account.withdraw(150)
except InsufficientBalanceError as e:
    print('业务异常：', e)


业务异常： 余额不足，无法取款


### 解释

- 自定义异常通常继承 `Exception`。
- 异常类名一般以 `Error` 结尾。
- 自定义异常可以让业务错误更加清晰，例如余额不足、库存不足、权限不足。


## 7. 查看异常堆栈

排查复杂错误时，只看错误信息可能不够，还需要查看异常发生的位置和调用链。


In [7]:
import traceback


def parse_number(text):
    return int(text)


def handle_data(text):
    return parse_number(text) * 2


try:
    handle_data('abc')
except ValueError:
    print('捕获到 ValueError')
    traceback.print_exc()


捕获到 ValueError


Traceback (most recent call last):
  File "C:\Users\11435\AppData\Local\Temp\ipykernel_1124\3582961644.py", line 13, in <module>
    handle_data('abc')
    ~~~~~~~~~~~^^^^^^^
  File "C:\Users\11435\AppData\Local\Temp\ipykernel_1124\3582961644.py", line 9, in handle_data
    return parse_number(text) * 2
           ~~~~~~~~~~~~^^^^^^
  File "C:\Users\11435\AppData\Local\Temp\ipykernel_1124\3582961644.py", line 5, in parse_number
    return int(text)
ValueError: invalid literal for int() with base 10: 'abc'


### 解释

- `traceback.print_exc()` 会打印完整异常堆栈。
- 堆栈信息可以帮助定位异常从哪里开始、经过了哪些函数。
- 实际项目中也常用 `logging.exception()` 记录异常堆栈。


## 8. 异常处理建议

1. 只捕获你能处理的异常。
2. 不要把所有代码都放进一个巨大的 `try` 中。
3. 不要裸写 `except:`，它会捕获太多问题。
4. 捕获异常后要么处理，要么记录，不要悄悄吞掉。
5. 参数非法时可以用 `raise` 明确告诉调用者。
6. 文件、网络、锁等资源清理适合放到 `finally` 或 `with` 中。
7. 异常处理不是逻辑判断的替代品；能用普通判断清楚处理的情况，不必强行用异常。
